In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Compatible si le notebook est lance depuis la racine, test_cf/ ou test_cf/notebooks/.
ROOT = Path.cwd()
if (ROOT / "core").is_dir():
    TEST_CF = ROOT
elif ROOT.name == "notebooks" and (ROOT.parent / "core").is_dir():
    TEST_CF = ROOT.parent
else:
    TEST_CF = ROOT / "test_cf"
BIEN_DIR = TEST_CF / "measurements" / "real" / "bien"
OUT_DIR = TEST_CF / "assets" / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_FILES = [BIEN_DIR / f"{i}real_run.csv" for i in range(1, 5)]


def _load_runs(file_list):
    all_data = []
    for f in file_list:
        if f.exists():
            df = pd.read_csv(f).sort_values("time").reset_index(drop=True)
            all_data.append(df)
        else:
            print(f"[skip] fichier absent: {f}")
    if not all_data:
        raise FileNotFoundError(f"Aucun fichier trouve dans {BIEN_DIR}")
    return all_data


def _event_times(df, event_name):
    if "event" not in df.columns:
        return []
    mask = df["event"].fillna("").astype(str).eq(event_name)
    return df.loc[mask, "time"].astype(float).tolist()


def _event_summary(all_data):
    rows = []
    for i, df in enumerate(all_data, start=1):
        triggers = _event_times(df, "finetune_trigger")
        swaps = _event_times(df, "hotswap")
        compute = np.nan
        if swaps and "compute_time_s" in df.columns:
            vals = pd.to_numeric(
                df.loc[df["event"].fillna("").astype(str).eq("hotswap"), "compute_time_s"],
                errors="coerce",
            ).dropna()
            if len(vals):
                compute = float(vals.iloc[0])
        rows.append({
            "vol": i,
            "debut_mesure_s": triggers[0] if triggers else np.nan,
            "swap_s": swaps[0] if swaps else np.nan,
            "calcul_s": compute,
            "duree_s": float(df["time"].max()),
        })
    summary = pd.DataFrame(rows)
    try:
        display(summary)
    except NameError:
        print(summary.to_string(index=False))
    return summary


def _add_event_markers(axs, all_data):
    trigger_times = []
    swap_times = []
    for df in all_data:
        trigger_times.extend(_event_times(df, "finetune_trigger"))
        swap_times.extend(_event_times(df, "hotswap"))

    trigger_med = float(np.median(trigger_times)) if trigger_times else None
    swap_med = float(np.median(swap_times)) if swap_times else None

    for ax in axs:
        # Lignes fines pour chaque vol, puis ligne mediane plus lisible.
        for t in trigger_times:
            ax.axvline(t, color="steelblue", lw=0.8, alpha=0.18)
        for t in swap_times:
            ax.axvline(t, color="navy", lw=0.8, alpha=0.18)

        if trigger_med is not None:
            ax.axvline(trigger_med, color="steelblue", lw=1.8, linestyle="--", alpha=0.95,
                       label="debut mesure / finetune")
        if swap_med is not None:
            ax.axvline(swap_med, color="navy", lw=1.8, linestyle="--", alpha=0.95,
                       label="hot-swap")


def plot_drone_envelope(file_list=RUN_FILES,
                        title="Trajectoires reelles et enveloppes - 5 vols "):
    all_data = _load_runs(file_list)
    summary = _event_summary(all_data)

    # Base de temps commune, comme dans la version precedente.
    min_duration = min(df["time"].max() for df in all_data)
    common_time = np.linspace(0, min_duration, 1000)

    fig, axs = plt.subplots(
        3, 1, figsize=(10, 12), sharex=True,
        gridspec_kw={"height_ratios": [1, 1, 0.7]},
    )

    # Meme ordre et memes couleurs que ta version precedente.
    # Seule la fenetre Z est adaptee aux nouvelles mesures autour de 0.5 m.
    plot_configs = [
        (["x_pos", "y_pos"], ["blue", "dodgerblue"], "Position XY (m)",
         "Positions Horizontales", [-0.35, 0.55]),
        (["roll", "pitch"], ["blue", "dodgerblue"], "Angles (deg)",
         "Attitude et Oscillations", [-10, 10]),
        (["z_pos"], ["blue"], "Altitude Z (m)",
         "Stabilite Verticale", [0.0, 0.7]),
    ]

    for i, (keys, colors, ylabel, subtitle, ylim) in enumerate(plot_configs):
        for key, color in zip(keys, colors):
            interp_list = []
            for df in all_data:
                if key in df.columns:
                    interp_list.append(np.interp(common_time, df["time"], df[key]))

            if not interp_list:
                continue

            matrix = np.array(interp_list)
            mean_val = np.mean(matrix, axis=0)
            min_val = np.min(matrix, axis=0)
            max_val = np.max(matrix, axis=0)

            # Meme representation: enveloppe min/max + moyenne.
            axs[i].fill_between(common_time, min_val, max_val, color=color, alpha=0.15)
            axs[i].plot(common_time, mean_val, color=color, lw=2,
                        label=f'{key.split("_")[0].upper()} Moyenne')

            cmd_key = f"{key}_cmd"
            if cmd_key in all_data[0].columns:
                cmd_interp = np.interp(common_time, all_data[0]["time"], all_data[0][cmd_key])
                axs[i].plot(common_time, cmd_interp, color="black", linestyle="--", alpha=0.6,
                            label=f"{key[0].upper()} Consigne")

        axs[i].set_ylabel(ylabel)
        axs[i].set_ylim(ylim)
        axs[i].set_title(subtitle)
        axs[i].grid(True, which="both", linestyle="--", alpha=0.5)

    _add_event_markers(axs, all_data)

    # Meme placement general des legendes que la version precedente.
    for i, ax in enumerate(axs):
        handles, labels = ax.get_legend_handles_labels()
        dedup = dict(zip(labels, handles))
        loc = "lower right" if i in (0, 2) else "upper right"
        ax.legend(dedup.values(), dedup.keys(), loc=loc, fontsize="x-small", ncol=2)

    axs[-1].set_xlabel("Temps (s)")
    plt.suptitle(title, fontsize=14, fontweight="bold")
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])

    out = OUT_DIR / "bien_4_vols_enveloppe_events.png"
    plt.savefig(out, dpi=300)
    print(f"[save] {out}")
    plt.show()
    return summary


summary = plot_drone_envelope()
